# Panel job-4 v3 — one pretrain, parallel sharded embedding

Use the SAME frozen encoder for every session. Set `RUN_PRETRAIN=True` in **session 0 only**; set it to `False` in every other session. Other sessions extract their local panel copy and wait safely for `encoder_final.pt`, then embed their disjoint shards concurrently.

Example: session 0 `SHARD=0, RUN_PRETRAIN=True`; session 1 `SHARD=1, RUN_PRETRAIN=False`; session 2 `SHARD=2, RUN_PRETRAIN=False`. Set `NSHARDS=3` in all three. A100 is preferred for session 0; any CUDA GPU works for embedding-only sessions.

Resource use: pretraining keeps the registered batch=512/LR schedule, with automatic vCPU workers, prefetch and bf16. Embedding defaults to host-RAM-safe batch=512, workers<=4, prefetch=2 (about 2.3 GiB queued RGB tensor payload), with optional batch override and CUDA-OOM halving. This fixes the killed-worker failure from oversized batch x worker x prefetch queues.

Resume-safe: atomic epoch checkpoints, deterministic 50k-row shard parts, per-part sha256 manifests. Existing parts are verified and skipped. The last finished shard merges all manifests into `emb_panels_manifest.json`, the original home-judge input. Requires job-3 `FARM_COMPLETE.txt`; no partial result is judge-safe. Run all cells; each runtime unassigns after its own shard finishes.

In [ ]:
RUN_PRETRAIN = True  # True in session 0 ONLY; False in every other session
SHARD = 0            # unique per session: 0, 1, 2, ...
NSHARDS = 3          # SAME in every session
EMBED_BATCH = 512    # host-RAM-safe default; positive integer overrides
WORKERS = 4          # bound queued image tensors; -1 auto also caps embed at 4
DRIVE_PANELS = "/content/drive/MyDrive/brain_packs/panel_farm"
OUT_DIR = "/content/drive/MyDrive/brain_packs/panel_embed"
RAW = "https://raw.githubusercontent.com/vicjayjay/brain-packs/main/code/"
print(f"shard {SHARD}/{NSHARDS}; pretrain={RUN_PRETRAIN}")

In [ ]:
import subprocess, sys, urllib.request
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "pandas", "pyarrow", "torchvision"], check=True)
from google.colab import drive  # type: ignore
drive.mount("/content/drive")
dst = Path("/content/panel_simclr_colab.py")
if dst.exists():
    dst.unlink()
urllib.request.urlretrieve(RAW + "panel_simclr_colab.py?v=4", dst)
sys.path.insert(0, "/content")
import torch
assert torch.cuda.is_available(), "needs a CUDA GPU runtime"
print(torch.cuda.get_device_name(0), "— pack + code ready")

In [ ]:
import importlib, sys
import panel_simclr_colab as sim
importlib.reload(sim)
if RUN_PRETRAIN:
    assert SHARD == 0, "only session 0 may own pretraining"
    sys.argv = ["simclr", "pretrain", "--drive-panels", DRIVE_PANELS, "--out", OUT_DIR]
    sim.main()
else:
    print("embedding-only session: will wait for session 0's frozen encoder")

In [ ]:
import importlib, sys
import panel_simclr_colab as sim
importlib.reload(sim)
sys.argv = ["simclr", "embed", "--drive-panels", DRIVE_PANELS, "--out", OUT_DIR,
            "--shard", str(SHARD), "--nshards", str(NSHARDS),
            "--embed-batch", str(EMBED_BATCH), "--workers", str(WORKERS)]
sim.main()

In [ ]:
from pathlib import Path
combined = Path(OUT_DIR) / "emb_panels_manifest.json"
print("JUDGE INPUT READY" if combined.exists() else "shard saved; other shard(s) still pending")
from google.colab import runtime  # type: ignore
runtime.unassign()